# 章节实践

通过本章的系统学习，我们已经掌握了使用高阶API开发矩阵算子的完整方法。为了巩固所学知识，现提供以下实践练习：实现Ascend C算子Matmul,算子命名为MatmulCustom，编写其kernel侧代码、host侧代码，并完成aclnn调用测试。
相关算法：

                                  C = A * B + Bias

支持1024， K = 256， N = 640的int8类型输入，int32类型输出，算子原型为：

<div style="width: 100%;">
<table style="border-collapse: collapse; width: 550px; max-width: 550px; border: 1px solid #ddd; margin-left: 0 !important;">
  <!-- 表头行：算子类型 -->
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子类型 (OpType)</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > MatmulCustom</td>
  </tr>
  
  <!-- 算子输入模块：纵向合并单元格 -->
  <tr>
    <!-- 算子输入 纵向合并3行（x行+y行+列名行） -->
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1024, 256)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">int8</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(256, 640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">int8</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
   <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">int32</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr> 
  <!-- 算子输出模块：纵向合并单元格 -->

  <tr>
  <td  style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(1024, 640)</td>
    <td style="padding: 8px;">int32</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

要求：

- 完成MatmulCustom算子host侧核函数调用函数相关代码补齐。

- 完成MatmulCustom算子kernel侧核函数实现相关代码补齐。

- 支持输入类型为int8，输出类型为int32。

请开始你的实践，体验从理解到创造的完整开发过程。

---

## 环境准备：

In [9]:
import os
!mkdir -p Sources
!bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env' > /tmp/shell_env.txt  # bash
with open("/tmp/shell_env.txt", "r") as f:
    for line in f.readlines():
        line = line.strip()
        if "=" in line and not line.startswith(("#", " ")):
            key, value = line.split("=", 1)
            os.environ[key] = value
print("\n🎉 Environment initialization process completed successfully!")

## 生成算子工程

执行如下命令准备创建工具msopgen

In [ ]:
# 下载工具仓代码
!git clone https://gitcode.com/Ascend/msopgen.git
#编译安装msopgen工具
!cd msopgen; pip install -r requirements.txt;python build.py;cd output;pip install mindstudio_opgen*.whl --force-reinstall;pip install mindstudio_opst*.whl --force-reinstall


---

## 算子原型json文件准备

该部分不需要修改，直接执行即可。

In [ ]:
%%writefile Sources/MatmulCustom.json

[
    {
        "op": "MatmulCustom",
        "language": "cpp",
        "input_desc": [
            {
                "name": "a",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "int8"
                ]
            },
            {
                "name": "b",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "int8"
                ]
            },
            {
                "name": "bias",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "int32"
                ]
            }
        ],
        "output_desc": [
            {
                "name": "c",
                "param_type": "required",
                "format": [
                    "ND"
                ],
                "type": [
                    "int32"
                ]
            }
        ]
    }
]

---

## 算子工程创建

该部分不需要修改，直接执行即可。

In [ ]:
# 清除已有的custom_op目录
!rm -rf Sources/custom_op
# 使用msopgen工具创建算子工程前设置使用开源仓编译的msopgen工具，不设置时默认使用CANN安装目录下的msopgen工具
!export PATH=/usr/local/bin:~/.local/bin:$PATH;msopgen gen -i Sources/MatmulCustom.json -c ai_core-ascend910b1 -lan cpp -out Sources/custom_op

---

## Tiling结构体定义修改

修改代码，完成Tiling结构体定义

In [ ]:
%%writefile Sources/custom_op/op_host/matmul_custom_tiling.h

#include "register/tilingdata_base.h"
#include "tiling/tiling_api.h"

using namespace matmul_tiling;
namespace optiling {
BEGIN_TILING_DATA_DEF(MatmulCustomTilingData)
  TILING_DATA_FIELD_DEF(uint32_t, size);
END_TILING_DATA_DEF;

REGISTER_TILING_DATA_CLASS(MatmulCustom, MatmulCustomTilingData)
}


---

## host侧代码修改

修改代码，完成Tiling实现

In [ ]:
%%writefile Sources/custom_op/op_host/matmul_custom.cpp



---

## Kernel实现修改

修改代码，完成Kernel实现

In [ ]:
%%writefile Sources/custom_op/op_kernel/matmul_custom.cpp

#include "kernel_operator.h"
#include "lib/matmul_intf.h"
using namespace matmul;

extern "C" __global__ __aicore__ void matmul_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR workspace, GM_ADDR tiling) {
    GET_TILING_DATA(tiling_data, tiling);
    // TODO: user kernel impl
}

---
## 部署算子运行测试

修改完成后，执行如下命令部署算子运行测试。

In [ ]:
%%writefile Sources/aclnn_test.cpp

#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_matmul_custom.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)

int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));

    // 定义MatmulLeakyrelu的张量形状
    const std::vector<int64_t> shape_a = {1024, 256};    // 输入a形状
    const std::vector<int64_t> shape_b = {256, 640};     // 输入b形状
    const std::vector<int64_t> shape_bias = {640};       // 输入bias形状
    const std::vector<int64_t> shape_output = {1024, 640};// 输出形状

    // 计算各张量元素数量和内存大小
    const int64_t elementCount_a = shape_a[0] * shape_a[1];
    const int64_t elementCount_b = shape_b[0] * shape_b[1];
    const int64_t elementCount_bias = shape_bias[0];
    const int64_t elementCount_output = shape_output[0] * shape_output[1];
    
    const size_t bufferSize_a = elementCount_a * sizeof(int8_t);
    const size_t bufferSize_b = elementCount_b * sizeof(int8_t);
    const size_t bufferSize_bias = elementCount_bias * sizeof(int32_t);
    const size_t bufferSize_output = elementCount_output * sizeof(int32_t);

    // 分配输入a的设备内存并创建张量
    void* inputADeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputADeviceMem, bufferSize_a, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputA = aclCreateTensor(shape_a.data(), shape_a.size(), ACL_INT8, nullptr, 0, ACL_FORMAT_ND,
                                        shape_a.data(), shape_a.size(), inputADeviceMem);

    // 分配输入b的设备内存并创建张量
    void* inputBDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBDeviceMem, bufferSize_b, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputB = aclCreateTensor(shape_b.data(), shape_b.size(), ACL_INT8, nullptr, 0, ACL_FORMAT_ND,
                                        shape_b.data(), shape_b.size(), inputBDeviceMem);

    // 分配输入bias的设备内存并创建张量
    void* inputBiasDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBiasDeviceMem, bufferSize_bias, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputBias = aclCreateTensor(shape_bias.data(), shape_bias.size(), ACL_INT32, nullptr, 0, ACL_FORMAT_ND,
                                           shape_bias.data(), shape_bias.size(), inputBiasDeviceMem);

    // 分配输出的设备内存并创建张量
    void* outputDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&outputDeviceMem, bufferSize_output, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output = aclCreateTensor(shape_output.data(), shape_output.size(), ACL_INT32, nullptr, 0, ACL_FORMAT_ND,
                                        shape_output.data(), shape_output.size(), outputDeviceMem);

    // 准备主机测试数据
    std::vector<int8_t> inputAHostData(elementCount_a, 1);
    std::vector<int8_t> inputBHostData(elementCount_b, 2);
    std::vector<int32_t> inputBiasHostData(elementCount_bias, 10);
    std::vector<int32_t> outputHostData(elementCount_output, 0);
    
    // 计算预期结果
    std::vector<int32_t> goldenData(elementCount_output, 522);

    // 主机数据拷贝到设备
    CHECK_ACL(aclrtMemcpy(inputADeviceMem, bufferSize_a, inputAHostData.data(),
                          bufferSize_a, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBDeviceMem, bufferSize_b, inputBHostData.data(),
                          bufferSize_b, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBiasDeviceMem, bufferSize_bias, inputBiasHostData.data(),
                          bufferSize_bias, ACL_MEMCPY_HOST_TO_DEVICE));

    // 获取Matmul算子工作空间大小和执行器
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnMatmulCustomGetWorkspaceSize(inputA, inputB, inputBias, output, &workspaceSize, &executor));

    // 分配工作空间内存
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }

    // 执行Matmul算子
    CHECK_ACL(aclnnMatmulCustom(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));

    // 设备结果拷贝回主机
    CHECK_ACL(aclrtMemcpy(outputHostData.data(), bufferSize_output, outputDeviceMem,
                          bufferSize_output, ACL_MEMCPY_DEVICE_TO_HOST));

    // 打印结果并验证
    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount_output, 10);
    for (int64_t i = 0; i < previewCount; i++) { 
        printf("%d ", outputHostData[i]); 
    }
    printf("\ntest %s\n", std::equal(outputHostData.begin(), outputHostData.end(), goldenData.begin()) ? "pass" : "failed");

    // 释放资源
    aclDestroyTensor(inputA);
    aclDestroyTensor(inputB);
    aclDestroyTensor(inputBias);
    aclDestroyTensor(output);
    
    CHECK_ACL(aclrtFree(inputADeviceMem));
    CHECK_ACL(aclrtFree(inputBDeviceMem));
    CHECK_ACL(aclrtFree(inputBiasDeviceMem));
    CHECK_ACL(aclrtFree(outputDeviceMem));
    
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    
    return 0;
}

In [ ]:
#编译算子工程并部署编译出的算子包
!cd Sources/custom_op;bash build.sh;
!./Sources/custom_op/build_out/custom_opp*.run --install-path=${HOME}/

In [ ]:
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/aclnn_test.cpp -lcust_opapi -lnnopbase -lacl_rt -o execute_add_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./execute_add_op

预期执行效果如下：

<img src="./images/结果图片05.png" alt="结果图片05"  width="500px" >

### 执行以下代码获取答案。

Tiling结构体定义

In [ ]:
!cat ./answer/04.04_answer/op_host/matmul_custom_tiling.h

host侧实现

In [ ]:
!cat ./answer/04.04_answer/op_host/matmul_custom.cpp

kernel实现

In [ ]:
!cat ./answer/04.04_answer/op_kernel/matmul_custom.cpp